### This notebook demonstrates how to convert a PyTorch model to TensorFlow Lite.

In [1]:
import torch
import sys
import os
import tensorflow as tf
from ai_edge_quantizer import quantizer, recipe

from ai_edge_litert.interpreter import Interpreter
from ai_edge_quantizer.utils import tfl_interpreter_utils
import litert_torch

sys.path.append(os.path.abspath(".."))
from models.murenn_layer import MuReNNLayer
from data.yellowhammer import TrainingDataset

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.1).
W0415 14:12:28.061000 99210 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Load the PyTorch model

In [23]:
model = MuReNNLayer(J=2, Q=16, T=32, in_channels=1, J_phi=6, use_conv1d=True, use_power=True)
model.eval()

MuReNNLayer(
  (dtcwt): UDTCWTDirect(
    (fwd_j1): FWD_J1()
    (fwd_j2plus): ModuleList(
      (0): FWD_J2PLUS()
    )
  )
  (conv1d): ParameterList(
      (0): Object of type: Conv1d
      (1): Object of type: Conv1d
    (0): Conv1d(1, 16, kernel_size=(32,), stride=(1,), padding=same, bias=False)
    (1): Conv1d(1, 16, kernel_size=(32,), stride=(1,), padding=same, bias=False)
  )
  (down): Downsampling(
    (phi): DTCWTDirect()
    (relu): ReLU()
  )
  (root): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x16x1]
      (1): Parameter containing: [torch.float32 of size 1x16x1]
  )
  (sigmoid): Sigmoid()
)

#### Convert Pytorch model to Keras model

In [24]:
dummy_input = (torch.randn(1, 1, 30720),)
tflite_model = litert_torch.convert(model, dummy_input)
tflite_model.export("murenn_layer.tflite")

INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpxbbnlj8d/assets


INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpxbbnlj8d/assets
W0000 00:00:1775829058.097344 32249911 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1775829058.097356 32249911 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1775829058.097580 32249911 reader.cc:83] Reading SavedModel from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpxbbnlj8d
I0000 00:00:1775829058.098092 32249911 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1775829058.098095 32249911 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpxbbnlj8d
I0000 00:00:1775829058.103435 32249911 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1775829058.126392 32249911 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpxbbnlj8d
I0000 00:00:1775829058.133480 32249911 loader.

### Convert the Keras model to TensorFlow Lite

In [25]:
interpreter = Interpreter(model_path="murenn_layer.tflite")
input_details = interpreter.get_input_details()

In [26]:
import re
def clean_input_layer_name(name: str) -> str:
    name = name.split(":")[0]
    name = re.sub(r"^serving_default_", "", name)
    return name

data_dir = '/Users/zhang/MuReNN/YH_data_with_aug/train'
dataset = TrainingDataset(data_dir)
# Randomly select 1 samples for calibration
idx = torch.randperm(len(dataset))[:1]
dataset = [dataset[i] for i in idx]
calibration_samples = []

for i, sample in enumerate(dataset):
    calibration_samples.append({
        clean_input_layer_name(input_details[0]['name']): sample['waveform'].reshape(1, 1, -1),
    })  

calibration_data = {
    tfl_interpreter_utils.DEFAULT_SIGNATURE_KEY: calibration_samples,
}

In [28]:
qt_static = quantizer.Quantizer("murenn_layer.tflite")
qt_static.load_quantization_recipe(recipe.static_wi8_ai16())
calibration_result = qt_static.calibrate(calibration_data)
qt_static.quantize(calibration_result).export_model("dummy_murennINT16.tflite", overwrite=True)